<a href="https://colab.research.google.com/github/Bonesha-5/Group4_Formative3_Assignment/blob/main/gradient_desent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Gradient Descent in Code
This notebook turns our hand calculations from Part 3 into Python.

**Model:** y_hat = X · m + b  
**Cost (MSE):** J = (1/n) · Σ(y_hat − y)²  
**Update:** m = m − α · ∂J/∂m and b = b − α · ∂J/∂b

## 1. Imports

In [13]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.differentiate import derivative   # needs SciPy 1.15 or newer

## 2. Data as Matrices

Each **row** of X is one data point. m, b and y are **columns** (2 × 1).  
b is one shared bias, written as a column so it can be added to every row.

In [14]:
X = np.array([[1.0,  3.0],
              [4.0, 10.0]])
y = np.array([[5.0],
              [6.0]])

m_start = np.array([[-1.0],
                    [ 2.0]])
b_start = 1.0                   # one shared bias (b · [1, 1])

ones = np.ones((2, 1))          # column of ones: turns b into [b, b]
n = X.shape[0]                  # number of data points = 2

LEARNING_RATE = 0.01
N_ITERATIONS = 20

print("n =", n)

n = 2


## 3. Small Functions (Each one job)

In [15]:
def predict(X, m, b):
    """ŷ = X · m + b·[1, 1]"""
    return X @ m + b * ones


def compute_errors(y_hat, y):
    """e = ŷ − y"""
    return y_hat - y


def mse(errors):
    """J = (1/n) · eᵀ · e"""
    return (errors.T @ errors).item() / n

## 4: The equation, and SciPy to take its derivative

`cost_equation` is our MSE written as one equation of the parameters [m1, m2, b].

`scipy_derivative` **accepts any equation** and returns its derivative with respect to one parameter. It works like this:

- It keeps all parameters fixed except one.
- It lets SciPy move that one parameter slightly to measure the slope.

In [16]:
def cost_equation(params):
    """The MSE as an equation of params = [m1, m2, b]."""
    m = np.array([[params[0]],
                  [params[1]]])
    b = params[2]
    return mse(compute_errors(predict(X, m, b), y))


def scipy_derivative(equation, params, index):
    """Derivative of `equation` with respect to params[index], using SciPy."""
    def equation_of_one_param(value):
        new_params = list(params)
        new_params[index] = value
        return equation(new_params)

    # SciPy sends in many values at once, so we handle them one by one
    vectorized = np.vectorize(equation_of_one_param)
    result = derivative(vectorized, params[index])
    return float(result.df)


# Quick test on a simple equation: f(x) = x², so f'(3) should be 6
print("Test, derivative of x² at x = 3:", round(scipy_derivative(lambda p: p[0] ** 2, [3.0], 0), 6))

Test, derivative of x² at x = 3: 6.0


### Step 5: The gradients

We get the gradients from SciPy, and we also compute them with our matrix formulas from Part 3 as a check:

- $\partial J/\partial m = (2/n)\cdot X^T\cdot e$
- $\partial J/\partial b = (2/n)\cdot \mathbf{1}^T\cdot e$

In [17]:
def gradients_with_scipy(m, b):
    params = [m[0, 0], m[1, 0], b]
    dJ_dm1 = scipy_derivative(cost_equation, params, 0)
    dJ_dm2 = scipy_derivative(cost_equation, params, 1)
    dJ_db  = scipy_derivative(cost_equation, params, 2)
    return np.array([[dJ_dm1], [dJ_dm2]]), dJ_db


def gradients_with_matrices(errors):
    dJ_dm = (2 / n) * (X.T @ errors)
    dJ_db = (2 / n) * (ones.T @ errors).item()
    return dJ_dm, dJ_db


def update(m, b, dJ_dm, dJ_db, learning_rate):
    new_m = m - learning_rate * dJ_dm
    new_b = b - learning_rate * dJ_db
    return new_m, new_b

### Step 6: Run gradient descent

Every step is printed so you can follow it. We save m, b and the error after each iteration for the plots.

In [18]:
def run_gradient_descent(m, b, learning_rate, n_iterations, show_steps=3):
    history = {"m1": [m[0, 0]], "m2": [m[1, 0]], "b": [b], "error": []}

    for i in range(1, n_iterations + 1):
        y_hat = predict(X, m, b)
        errors = compute_errors(y_hat, y)
        error = mse(errors)
        dJ_dm, dJ_db = gradients_with_scipy(m, b)
        check_dm, check_db = gradients_with_matrices(errors)

        if i <= show_steps:
            print(f"========== Iteration {i} ==========")
            print("m =", m.ravel().round(6), "  b =", round(b, 6))
            print("y_hat =", y_hat.ravel().round(6))
            print("y difference =", errors.ravel().round(6))
            print("MSE =", round(error, 6))
            print("∂J/∂m (SciPy formula)  =", dJ_dm.ravel().round(6), "  ∂J/∂b (SciPy formula)  =", round(dJ_db, 6))
            print("∂J/∂m (handwritten formula) =", check_dm.ravel().round(6), "  ∂J/∂b (handwritten formula) =", round(check_db, 6))

        m, b = update(m, b, dJ_dm, dJ_db, learning_rate)

        if i <= show_steps:
            print("new m =", m.ravel().round(6), "  new b =", round(b, 6), "\n")

        history["m1"].append(m[0, 0])
        history["m2"].append(m[1, 0])
        history["b"].append(b)
        history["error"].append(error)

    history["error"].append(mse(compute_errors(predict(X, m, b), y)))
    return m, b, history


m_final, b_final, history = run_gradient_descent(m_start, b_start, LEARNING_RATE, N_ITERATIONS)

========== Iteration 1 ==========
m = [-1.  2.]   b = 1.0
y_hat = [ 6. 17.]
y difference = [ 1. 11.]
MSE = 61.0
∂J/∂m (SciPy formula)  = [ 45. 113.]   ∂J/∂b (SciPy formula)  = 12.0
∂J/∂m (handwritten formula) = [ 45. 113.]   ∂J/∂b (handwritten formula) = 12.0
new m = [-1.45  0.87]   new b = 0.88 

========== Iteration 2 ==========
m = [-1.45  0.87]   b = 0.88
y_hat = [2.04 3.78]
y difference = [-2.96 -2.22]
MSE = 6.845
∂J/∂m (SciPy formula)  = [-11.84 -31.08]   ∂J/∂b (SciPy formula)  = -5.18
∂J/∂m (handwritten formula) = [-11.84 -31.08]   ∂J/∂b (handwritten formula) = -5.18
new m = [-1.3316  1.1808]   new b = 0.9318 

========== Iteration 3 ==========
m = [-1.3316  1.1808]   b = 0.9318
y_hat = [3.1426 7.4134]
y difference = [-1.8574  1.4134]
MSE = 2.723817
∂J/∂m (SciPy formula)  = [3.7962 8.5618]   ∂J/∂b (SciPy formula)  = -0.444
∂J/∂m (handwritten formula) = [3.7962 8.5618]   ∂J/∂b (handwritten formula) = -0.444
new m = [-1.369562  1.095182]   new b = 0.93624 



### Step 7: Check against our hand calculations

The first 3 iterations should match Part 3: m = [-1.369562, 1.095182], b = 0.93624.

In [20]:
# Display gradient descent results without Pandas

print(f"{'Iter':<6} {'m1':<12} {'m2':<12} {'b':<12} {'MSE':<12}")
print("-" * 58)

for i in range(len(history["m1"])):
    print(
        f"{i:<6} "
        f"{history['m1'][i]:<12.6f} "
        f"{history['m2'][i]:<12.6f} "
        f"{history['b'][i]:<12.6f} "
        f"{history['error'][i]:<12.6f}"
    )

Iter   m1           m2           b            MSE         
----------------------------------------------------------
0      -1.000000    2.000000     1.000000     61.000000   
1      -1.450000    0.870000     0.880000     6.845000    
2      -1.331600    1.180800     0.931800     2.723817    
3      -1.369562    1.095182     0.936240     2.390444    
4      -1.364477    1.118634     0.953620     2.344018    
5      -1.371209    1.112077     0.967349     2.319520    
6      -1.374664    1.113778     0.981992     2.296886    
7      -1.378996    1.113207     0.996294     2.274594    
8      -1.383060    1.113261     1.010601     2.252528    
9      -1.387172    1.113144     1.024818     2.230677    
10     -1.391246    1.113075     1.038972     2.209037    
11     -1.395305    1.112993     1.053055     2.187608    
12     -1.399342    1.112914     1.067070     2.166386    
13     -1.403361    1.112836     1.081017     2.145371    
14     -1.407360    1.112758     1.094896     2.124559  